In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
dataset=pd.read_csv("Social_Network_Ads.csv")

In [5]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [6]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [7]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [8]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [9]:
independent=dataset[['Age', 'EstimatedSalary',  'Gender_Male']]
dependent=dataset['Purchased']

In [11]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(independent,dependent,test_size=1/3,random_state=0)

In [12]:
from sklearn.preprocessing import StandardScaler
sc= StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [14]:
from sklearn.ensemble import RandomForestClassifier

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid= {'criterion':['gini','entropy'],
             'max_features':['auto','sqrt','log2'],
             'n_estimators':[10,100]}

grid=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1')

grid.fit(x_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\base.py", line 436, in _validate_params
 

GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_features': ['auto', 'sqrt', 'log2'],
                         'n_estimators': [10, 100]},
             scoring='f1', verbose=3)

In [17]:
re=grid.cv_results_
grid_predictions=grid.predict(x_test)

from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,grid_predictions)

from sklearn.metrics import classification_report
clf_report=classification_report(y_test,grid_predictions)

In [19]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter{}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter{'criterion': 'gini', 'max_features': 'log2', 'n_estimators': 100}: 0.9180821729616219


In [20]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[79  6]
 [ 5 44]]


In [21]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.94      0.93      0.93        85
           1       0.88      0.90      0.89        49

    accuracy                           0.92       134
   macro avg       0.91      0.91      0.91       134
weighted avg       0.92      0.92      0.92       134



In [22]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.9565426170468188)

In [23]:
table=pd.DataFrame.from_dict(re)

In [24]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003749,0.002403,0.000000,0.000000,gini,auto,10,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
1,0.003877,0.000459,0.000000,0.000000,gini,auto,100,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
2,0.054239,0.005714,0.012291,0.001958,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.764706,0.820513,0.829268,0.842105,0.918919,0.835102,0.049552,6
3,0.350048,0.017858,0.023046,0.003729,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.764706,0.850000,0.809524,0.926829,0.923077,0.854827,0.063309,2
4,0.047366,0.009988,0.013153,0.002271,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",0.727273,0.742857,0.731707,0.926829,0.882353,0.802204,0.084926,8
5,0.374544,0.052819,0.023161,0.003520,gini,log2,100,"{'criterion': 'gini', 'max_features': 'log2', ...",0.800000,0.842105,0.809524,0.926829,0.923077,0.860307,0.054612,1
6,0.002341,0.000284,0.000000,0.000000,entropy,auto,10,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
7,0.002239,0.000255,0.000000,0.000000,entropy,auto,100,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
8,0.053575,0.016382,0.011331,0.001858,entropy,sqrt,10,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.764706,0.810811,0.809524,0.871795,0.918919,0.835151,0.053984,5
9,0.338591,0.043319,0.024445,0.000628,entropy,sqrt,100,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.727273,0.871795,0.809524,0.900000,0.947368,0.851192,0.076353,4


In [25]:
age_input=float(input("Age:"))
estimatedsalary_input=float(input("EstimatedSalary:"))
gender_male_input=float(input("Gender_Male:"))

Age: 19
EstimatedSalary: 19000
Gender_Male: 1


In [26]:
Future_Prediction=grid.predict([[age_input,estimatedsalary_input,gender_male_input]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]
